# Hidreto de lítio $LiH$

Este notebook apresenta a simulação quântica da molécula de hidreto de lítio (LiH)
por meio do algoritmo Variational Quantum Eigensolver (VQE).

O experimento tem como objetivo estimar a energia do estado fundamental do sistema
molecular utilizando uma representação quântica do Hamiltoniano eletrônico.

In [ ]:
from src.hamiltonian import build_qubit_hamiltonian, get_atom_energy
from src.ansatz import build_ansatz
from src.optimizer import get_optimizer
from src.vqe_runner import run_vqe, run_experiment
from src.results import save_history, save_summary

import numpy as np

In [ ]:
from qiskit_algorithms.utils import algorithm_globals

algorithm_globals.random_seed = 137

## Definição do sistema molecular

Nesta etapa especificamos a geometria da molécula de hidreto de lítio.
Os átomos de lítio e hidrogênio são posicionados ao longo do eixo z,
com separação interatômica de 1.6 Å, valor utilizado como referência
para a construção inicial do sistema eletrônico.

In [ ]:
atom = "Li 0 0 0; H 0 0 1.6"

## Aplicando o VQE

Com a geometria molecular definida, realizamos o mapeamento do problema
eletrônico para um operador quântico em espaço de qubits. Esse Hamiltoniano
serve como entrada para o algoritmo VQE, responsável por aproximar
numericamente a energia fundamental do sistema.

In [ ]:
data = build_qubit_hamiltonian(atom)
ansatz = build_ansatz(name="efficient_su2", num_qubits=data["num_qubits"])
optimizer = get_optimizer(max_iter=200)

result = run_vqe(data["qubit_op"], ansatz, optimizer)

##  Persistência dos dados

Os resultados obtidos durante a otimização variacional são armazenados
para possibilitar análises posteriores e facilitar a reprodução do experimento
sob diferentes configurações ou parâmetros.

In [ ]:
save_summary(
    "lih_summary.csv",
    "LiH",
    result["energy"],
    data["fci_energy"],
    result["iterations"]
)

save_history("lih_history.csv", result["history"])


### Validando a saída

Por fim, avaliamos a energia estimada pelo algoritmo variacional,
permitindo verificar o comportamento do método aplicado ao sistema LiH.

In [ ]:
print("VQE:", result["energy"])
print("FCI:", data["fci_energy"])
print("Erro:", abs(result["energy"] - data["fci_energy"]))

## Curvas de energia

### Calcular energia do átomo H isolado

In [ ]:
E_H = get_atom_energy(atom_symbol="H", spin=1)
E_Li = get_atom_energy(atom_symbol="Li", spin=1)

E_ref = E_H + E_Li

### Plot da curva de dissociação

$$
E_{binding} = E(LiH) - (E(H) + E(Li))
$$

In [ ]:
distances = np.linspace(1.0, 4.0, 10)
distances = np.sort(distances)

vqe_energies = []
fci_energies = []

initial_point = None

for d in distances:
    config = {
        "molecule": "LiH",
        "geometry": f"Li 0 0 0; H 0 0 {d}",
        "basis": "sto-3g",
        "ansatz": "uccsd",
        "max_iter": 500,
    }

    result = run_experiment(config, initial_point)

    initial_point = result["optimal_params"]

    vqe_energies.append(result["energy"] - E_ref)
    fci_energies.append(result["fci"] - E_ref)

# Conversão Hartree para kcal/mol
hartree_to_kcal = 627.509

vqe_energies = np.array(vqe_energies) * hartree_to_kcal
fci_energies = np.array(fci_energies) * hartree_to_kcal

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(distances, vqe_energies, "o-", label="VQE")
plt.plot(distances, fci_energies, "s--", label="FCI")

plt.xlabel("Distância interatômica (Å)", fontsize=12)
plt.ylabel("Energia de dissociação (kcal/mol)", fontsize=12)
plt.title("Curva de dissociação do LiH - STO-3G", fontsize=14)

plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()